In [1]:
import torch
import pandas as pd

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

In [2]:
def reference_matmul(A, B):
    return A @ B

def quantize(x, scale):
    return torch.round(x / scale) * scale

def quantized_matmul(   A, B, scale = 0.1):
    A_q = quantize(A, scale)
    B_q = quantize(B, scale)
    return A_q @ B_q

In [3]:
A = torch.randn(256, 256)
B = torch.randn(256, 256)

ref = reference_matmul(A, B)
approx = quantized_matmul(A, B)

error = torch.norm(ref - approx) / torch.norm(ref)
print(error)

tensor(0.0407)


In [4]:
def evaluate(ref, approx):
    rel_error = torch.norm(ref - approx) / torch.norm(ref)
    max_error = torch.max(torch.abs(ref - approx))
    return pd.Series({
        "relative_error": rel_error.item(),
        "max_error": max_error.item()
    })

def fuzz_test(num_test = 100):
    results = []
    for _ in range(num_test):
        A = torch.randn(128, 128)
        B = torch.randn(128, 128)

        ref = A @ B
        approx = quantized_matmul(A, B, scale = 0.1)

        results.append(evaluate(ref, approx))
    
    return pd.concat(results, axis = 1). T

In [5]:
ref = A @ B
approx = quantized_matmul(A, B)

test1 = evaluate(ref, approx)
test2 = evaluate(ref, approx)

In [6]:
fuzz_test()

,relative_error,max_error
0,0.040645,2.011017
1,0.040307,2.124466
2,0.041192,2.006840
3,0.040441,1.843352
4,0.041001,1.867892
...,...,...
95,0.039904,1.902987
96,0.041241,1.983557
97,0.040882,1.894206
98,0.041228,2.068335
